# Prepare Crunchbase data for Mission Studio / Mission Radar

In [1]:
import pandas as pd

from src import PROJECT_DIR, logging

from discovery_utils.getters import crunchbase
from discovery_utils.utils.io import safe_yaml_load
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts
)

from discovery_utils.utils.llm import batch_check


PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}"

In [2]:
CB = crunchbase.CrunchbaseGetter()

2025-04-11 17:38:29,789 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-04-11 17:38:29,918 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2025-04-07


In [3]:
CONFIG_NAMES = [
    # "bioenergy",
    "biomass_heating",
    # "built_environment",
    "ccus",
    # "decarbonisation_general",
    "district_heating",
    # "energy_efficiency",
    # "energy_grid",
    # "energy_storage",
    "geothermal_energy",
    # "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    # "renewables_general",
    "solar_thermal",
    # "solar",
    # "wind"
]

## Run the LLM checks

In [8]:
def get_config_dict(config_name: str) -> dict:
    """Find companies in a specific category from a config file"""
    config_path = str(PROJECT_DIR / f"notebooks/{PROJECT_NAME}/config_{config_name}.yaml")
    return safe_yaml_load(open(config_path))

def get_companies_from_config(config: dict) -> pd.DataFrame:
    """Get companies from a config file"""
    category_name = config["search_recipe"]["category_name"]
    return CB.get_companies_in_nesta_categories("topic_labels", [category_name])

async def check_relevance(selected_df: pd.DataFrame, config_name: str, config: dict) -> None:
    """Check relevance of the selected companies"""
    selected_texts_df = CB.get_organisation_text(selected_df)
    check_data = dict(zip(selected_texts_df['id'], selected_texts_df['text']))
    # system_message = batch_check.generate_relevance_check_system_message(config)

    # fields = [
    #     {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    # ]
    system_message = batch_check.generate_relevance_check_system_message(config)
    system_message += """ 
    Mark the text as 'yes' if (one or more of the following):
    - If the technology defined by the scope above, is the main focus
    - If the technology is one of the components or activities described by the text. For example the technology could be  mentioned
        as part of a larger project or business including other technologies, or be mentioned as one of the use cases or case studies.
    - If the text describes a company and the technology is the main focus, or a part of a broader range of the company's activities and offerings.
    - If the text is about a component or critical element of the technology defined above
    - If the text describes a technology or process and explicitly mentions that it can be applied on the technology defined above to improve it's performance or efficiency.

    If the text is about heating technology but application target is not mentioned then assume it could be relevant for households or buildings (as opposed to an industrial applications).

    However, mark it as 'no' if (one or more of the following):
    - The activities, or business described in the text does not have a discernable impact on or connection with the technology.
    - If the technology is mentioned only in passing or as a minor example in a broader discussion, for example, 
        in only one sentence within a long text with many sentences, or at the very end of a long description.
    - The technology is mentioned only as a negative example (eg "unlike [technology]...")
    - The text mentions heat pumps for heating swimming pools  
    - The text would be better captured by one of the other categories (comma separated) mentioned in this list:  
    Bioenergy (biofuels), Biomass heating, Carbon capture and storage, District heating and heat networks, Energy grid, Geothermal energy, 
    Heat pumps, Hydrogen energy, Hydrogen heating, Micro CHP, Solar thermal heating, Energy storage (batteries), Solar power, Wind power
    """    
            
    fields = [
        {"name": "explanation", "type": "str", "description": "A short, 1-sentence explanation of the answer (max 25 words)."},
        {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},    
    ]    

    processor = batch_check.LLMProcessor(
        output_path=str(OUTPUT_DIR / f"llm_check_v2_{config_name}.jsonl"),
        system_message=system_message,
        session_name="mission_studio",
        output_fields=fields,
    )

    await processor.run(check_data, batch_size=10, sleep_time=0.5)    

In [9]:
async def check_all_configs(config_names) -> None:
    """Check relevance for all config files"""
    for config_name in config_names:
        logging.info(f"Checking relevance for {config_name}")
        config = get_config_dict(config_name)
        selected_df = get_companies_from_config(config)
        await check_relevance(selected_df, config_name, config)
        

In [11]:
await check_all_configs(CONFIG_NAMES)

2025-04-11 23:55:39,562 - root - INFO - Checking relevance for biomass_heating
2025-04-11 23:55:53,363 - root - INFO - Using OpenAI
2025-04-11 23:55:53,491 - root - INFO - All data has already been processed.
2025-04-11 23:55:53,492 - root - INFO - Checking relevance for ccus
2025-04-11 23:55:57,277 - root - INFO - Using OpenAI
2025-04-11 23:55:57,315 - root - INFO - All data has already been processed.
2025-04-11 23:55:57,316 - root - INFO - Checking relevance for district_heating
2025-04-11 23:56:00,807 - root - INFO - Using OpenAI
2025-04-11 23:56:00,841 - root - INFO - All data has already been processed.
2025-04-11 23:56:00,842 - root - INFO - Checking relevance for geothermal_energy
2025-04-11 23:56:04,187 - root - INFO - Using OpenAI
2025-04-11 23:56:04,219 - root - INFO - All data has already been processed.
2025-04-11 23:56:04,220 - root - INFO - Checking relevance for heat_pumps
2025-04-11 23:56:09,042 - root - INFO - Using OpenAI
2025-04-11 23:56:09,080 - root - INFO - All d

## Spot check the results

In [7]:
sheet_id = "1m9_tKyJDaSy2vDWxYVP_9HlfBbGysQUV-xrb1FW3vok"

In [15]:
import pandas as pd
from discovery_utils.utils import google

n_samples = 10
final_cols = ["theme", "id", "name", "text", "total_funding_gbp", "cb_url", "homepage_url", "is_relevant"]

llm_checks_df = []

for config_name in CONFIG_NAMES:
    # read a jsonl file
    llm_check_df = pd.read_json(OUTPUT_DIR / f"llm_check_MS_{config_name}.jsonl", lines=True)
    # get company descriptions
    org_texts = CB.get_organisation_text(llm_check_df)
    # sample n_samples from each group
    llm_check_df = (
        llm_check_df
        .merge(CB.organisations_enriched, left_on="id", right_on="id", how="left")
        .merge(org_texts, left_on="id", right_on="id", how="left")
        .assign(theme=config_name)
        .groupby("is_relevant")[final_cols]
        .apply(lambda df: df.sample(n_samples) if len(df) > n_samples else df)
        .reset_index(drop=True)
    )[final_cols]
    llm_checks_df.append(llm_check_df)

llm_checks_df = pd.concat(llm_checks_df, ignore_index=True)


In [ ]:
google.upload_data_to_gsheet(sheet_id, {"crunchbase_check_v2": llm_checks_df})
google.format_gsheet(sheet_id, "crunchbase_check_v2", freeze_cols=2)

2025-03-19 06:59:34,988 - root - INFO - Connected to Google Sheet: Mission Radar spot checks [2025-03-19]
2025-03-19 06:59:36,006 - root - INFO - Uploading DataFrame to sheet: crunchbase_check
/Users/karlis.kanders/Library/Caches/pypoetry/virtualenvs/discovery-mission-radar-prototyping-ejbE0IFh-py3.11/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Library/Caches/pypoetry/virtualenvs/discovery-mission-radar-prototyping-ejbE0IFh-py3.11/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-03-19 06:59:47,629 - root - INFO - Upload completed successfully.
2025-03-19 06:59:48,243 - root - INFO - Connected to Google Sheet: Mission Radar spot checks [2025-03-19]
